# **FINITE ELEMENTS in TIME in IRKSOME**

---

## **INSTALL**

### Firedrake

In [ ]:
try:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh"
    !bash "/tmp/firedrake-install.sh"
    from firedrake import *  # noqa: F401
except:
    from firedrake import *  # noqa: F401

### Irksome

In [ ]:
try:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git
    from irksome import *  # noqa: F401
except:
    from irksome import *  # noqa: F401

### Other

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

# Paper-style colours (matching the manuscript figures)
seabornblue   = "#4C72B0"
seaborngreen  = "#55A868"
seabornred    = "#C44E52"
seabornpurple = "#8172B2"
seabornorange = "#F4953B"

---

## **TIME STEPPING for ODEs**

*Apologies in advance to the audience of PDE people — we're going to start here.*

### General framework

Take an ODE for $x : [0, T] \to \mathbb{R}^n$:
> $$
> \dot{x} = f(x).
> $$

Suppose we have data $x_n \approx x(t_n)$ and want to advance to $t_{n+1} = t_n + \Delta t$.

A very general framework for an *$s$-stage* time discretisation is: seek a polynomial $x_h$ of degree $s$ on $[t_n, t_{n+1}]$ such that
1. $x_h(t_n) = x_n$ (interpolate the initial data); and
2. $s$ further constraints aim to capture the ODE $\dot{x} = f(x)$.

What are those $s$ further constraints?

### Constraint 1: Collocation $\to$ Runge–Kutta

Require that $\dot{x}_h = f(x_h)$ holds *exactly* at $s$ chosen *collocation points* $\xi_1, \ldots, \xi_s \in [t_n, t_{n+1}]$:
> $$
> \dot{x}_h(\xi_k) = f(x_h(\xi_k)), \qquad k = 1, \ldots, s.
> $$

This recovers the **collocation Runge–Kutta methods**:
- $s = 1$, midpoint node $\Rightarrow$ implicit midpoint.
- $s$ Gauss–Legendre nodes $\Rightarrow$ Gauss–Legendre RK (symplectic, order $2s$).
- $s$ right-Radau nodes $\Rightarrow$ Radau IIA (L-stable, order $2s - 1$).

### Constraint 2: Test against polynomial test functions $\to$ CPG

Putting our FEM hats on for a moment: rather than enforcing the ODE *pointwise* at $s$ nodes, we could enforce it *weakly* against polynomial test functions $\phi \in \mathcal{P}_{s-1}([t_n, t_{n+1}])$:
> $$
> \int_{t_n}^{t_{n+1}} (\dot{x}_h - f(x_h)) \, \phi \, \mathrm{d}t = 0, \qquad \forall \, \phi \in \mathcal{P}_{s-1}.
> $$

The mismatch in polynomial degrees (solution in $\mathcal{P}_s$, tests in $\mathcal{P}_{s-1}$) is by design: one of the solution's $s+1$ DoFs has already been spent on the initial condition, leaving $s$ unknowns to match the $s$-dimensional test space.

This defines the **continuous Petrov–Galerkin** (CPG) method:
- *Continuous*, because the solution function is continuous across timesteps (initial data is strongly enforced); and
- *Petrov–Galerkin*, because the trial and test spaces differ in polynomial degree.

### Why this is useful

**(i) CPG generalises collocation RK.** If we approximate the integral above using an $s$-node quadrature rule and take the collocation nodes as our quadrature points, we recover *exactly* the collocation RK method. So we don't lose anything by switching to CPG — but we gain the freedom of choosing other quadrature rules, with potentially better structure-preservation properties.

**(ii) Structure preservation.** This is where it gets interesting. For Hamiltonian systems with a *non-quadratic* Hamiltonian, CPG (with exact-in-time integration) exactly preserves the Hamiltonian. The symplectic Gauss–Legendre RK methods preserve quadratic invariants exactly but only preserve non-quadratic Hamiltonians approximately, with bounded drift. We'll see this in action in the demo below.

*(There's also a sibling FET scheme, discontinuous Galerkin (DG), which we won't discuss here but which is also implemented in Irksome.)*

---

## **DEMO: $N$-BODY HAMILTONIAN**

Take $N$ equal-mass particles in 2D moving under their own mutual gravity. The state at any time consists of positions $q_i \in \mathbb{R}^2$ and momenta $p_i \in \mathbb{R}^2$ for $i = 1, \ldots, N$, and the Hamiltonian (with a small softening parameter $\varepsilon$ to avoid singular close encounters) is
> $$
> H(q, p) \;=\; \sum_{i=1}^{N} \frac{|p_i|^2}{2m} \;-\; G\, m^2 \sum_{i<j} \frac{1}{\sqrt{|q_i - q_j|^2 + \varepsilon^2}}.
> $$

Hamilton's equations are
> $$
> \dot{q}_i = \frac{\partial H}{\partial p_i} = \frac{p_i}{m}, \qquad
> \dot{p}_i = -\frac{\partial H}{\partial q_i} = - G\, m^2 \sum_{j \neq i} \frac{q_i - q_j}{(|q_i - q_j|^2 + \varepsilon^2)^{3/2}}.
> $$

We'll set this up in Firedrake / Irksome, evolve it forward in time using both CPG and Gauss–Legendre schemes, and watch what each one does to the Hamiltonian over time.

### Setup

In [ ]:
# Number of particles
N = 5

# Constants
G    = Constant(1.0)
mass = Constant(1.0)
eps2 = Constant(0.05**2)  # softening, squared

# Irksome wants Firedrake `Function`s, so we use a trivial mesh and the
# "Real" element (globally constant DoFs). The Function holds 2N components,
# i.e. an N-particle 2D state, with no spatial dependence whatsoever.
mesh = UnitIntervalMesh(1)
Vq = VectorFunctionSpace(mesh, "R", 0, dim=2*N)
Vp = VectorFunctionSpace(mesh, "R", 0, dim=2*N)
Z = Vq * Vp

state = Function(Z)
q, p = split(state)
vq, vp = TestFunctions(Z)


def vec(field, i):
    """Extract the i-th particle's 2D sub-vector from a flat 2N-vector."""
    return as_vector([field[2*i], field[2*i+1]])

The Hamiltonian as a UFL expression, used for diagnostics:

In [ ]:
def kinetic(p_field):
    return sum(inner(vec(p_field, i), vec(p_field, i)) / (2 * mass)
               for i in range(N))

def potential(q_field):
    return - sum(G * mass**2
                 / sqrt(inner(vec(q_field, i) - vec(q_field, j),
                              vec(q_field, i) - vec(q_field, j)) + eps2)
                 for i in range(N) for j in range(i+1, N))

H_form = (kinetic(p) + potential(q)) * dx

Variational form encoding Hamilton's equations:

In [ ]:
# dq/dt - p/m = 0, tested against vq
F_integrand = inner(Dt(q) - p / mass, vq)

# dp/dt + dU/dq = 0, tested against vp
F_integrand += inner(Dt(p), vp)
for i in range(N):
    qi  = vec(q, i)
    vpi = vec(vp, i)
    for j in range(N):
        if j == i:
            continue
        diff = qi - vec(q, j)
        r2   = inner(diff, diff) + eps2
        F_integrand += G * mass**2 * inner(diff, vpi) / r2**1.5

F = F_integrand * dx

Initial conditions: positions and momenta drawn from a Gaussian, then momenta shifted so the total system momentum vanishes (so the centre of mass stays put).

In [ ]:
sigma_q = 1.0    # std dev of initial positions
sigma_p = 0.1    # std dev of initial momenta

def set_initial_conditions(seed=0):
    rng = np.random.default_rng(seed)
    q0 = rng.standard_normal(2*N) * sigma_q
    p0 = rng.standard_normal(2*N) * sigma_p
    # shift to zero total momentum
    p0 = p0.reshape(N, 2)
    p0 -= p0.mean(axis=0)
    p0 = p0.reshape(-1)
    state.sub(0).dat.data[:] = q0
    state.sub(1).dat.data[:] = p0

Pick a time stepping scheme. Switching between CPG and Gauss–Legendre is a one-character change — this is the headline ease-of-use feature.

In [ ]:
schemes = {
    "cpg": lambda deg: GalerkinCollocationScheme(deg, quadrature_degree="auto"),
    "gl":  GaussLegendre,
}

A small driver function that runs the integrator and records time series of energies and positions:

In [ ]:
solver_params = {
    "snes_type": "newtonls",
    "snes_rtol": 1e-12,
    "snes_atol": 1e-12,
    "ksp_type": "preonly",
    "pc_type": "lu",
}

def run(scheme_name, deg=2, T=50.0, Nt=2000, seed=0):
    set_initial_conditions(seed)

    scheme  = schemes[scheme_name](deg)
    dt      = Constant(T / Nt)
    t       = Constant(0.0)
    stepper = TimeStepper(F, scheme, t, dt, state,
                          solver_parameters=solver_params)

    times     = np.zeros(Nt + 1)
    positions = np.zeros((Nt + 1, 2*N))
    energies  = np.zeros(Nt + 1)

    positions[0] = state.sub(0).dat.data_ro.copy()
    energies[0]  = float(assemble(H_form))

    for k in range(Nt):
        stepper.advance()
        t.assign(float(t) + float(dt))
        times[k+1]     = float(t)
        positions[k+1] = state.sub(0).dat.data_ro.copy()
        energies[k+1]  = float(assemble(H_form))

    return {"times": times, "positions": positions, "energies": energies}

### Compare schemes

Run three schemes on identical initial conditions:
- **CPG(2)** — degree-2 continuous Petrov–Galerkin (preserves the Hamiltonian).
- **GL(1)** — 1-stage Gauss–Legendre, i.e. implicit midpoint (symplectic, but only preserves *quadratic* invariants exactly).
- **GL(2)** — 2-stage Gauss–Legendre at higher order, same story.

In [ ]:
results = {
    "cpg(2)": run("cpg", deg=2),
    "gl(1)":  run("gl",  deg=1),
    "gl(2)":  run("gl",  deg=2),
}

### Hamiltonian drift over time

If the integrator preserves $H$ exactly, the relative drift $(H(t) - H(0)) / |H(0)|$ should sit at solver tolerance. Otherwise we should see a drift signal — symplectic methods drift slowly and boundedly, but never to zero.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colours = {"cpg(2)": seabornblue, "gl(1)": seabornred, "gl(2)": seabornorange}

for name, res in results.items():
    rel = (res["energies"] - res["energies"][0]) / abs(res["energies"][0])
    ax.plot(res["times"], rel, label=name, color=colours[name], lw=1.5)

ax.set_xlabel("$t$")
ax.set_ylabel(r"$(H(t) - H(0)) \,/\, |H(0)|$")
ax.set_title("Relative Hamiltonian drift")
ax.axhline(0, color="grey", lw=0.5, ls="--")
ax.legend()
fig.tight_layout()
plt.show()

### Particle trajectories

Trajectories in the $(x, y)$ plane, coloured by time. One panel per integrator.

In [ ]:
def plot_orbits(ax, res, title):
    positions = res["positions"]
    times     = res["times"]
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(times.min(), times.max())
    for i in range(N):
        x = positions[:, 2*i]
        y = positions[:, 2*i+1]
        pts  = np.array([x, y]).T.reshape(-1, 1, 2)
        segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
        lc = LineCollection(segs, cmap=cmap, norm=norm, lw=0.5, alpha=0.85)
        lc.set_array(times[:-1])
        ax.add_collection(lc)
    all_x = positions[:, 0::2]
    all_y = positions[:, 1::2]
    pad = 0.1 * max(all_x.ptp(), all_y.ptp())
    ax.set_xlim(all_x.min() - pad, all_x.max() + pad)
    ax.set_ylim(all_y.min() - pad, all_y.max() + pad)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, name in zip(axes, results.keys()):
    plot_orbits(ax, results[name], name)
fig.tight_layout()
plt.show()

---

## **TO COME: PDE EXAMPLES**

*To be planned — examples on conservative (Hamiltonian-style) and dissipative (gradient-descent) PDE systems will go here, showing how the same one-line scheme switch carries over from ODEs to PDEs.*